In [16]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df = pd.read_csv("hf://datasets/AzharAli05/Resume-Screening-Dataset/dataset.csv")

In [17]:
df

,Role,Resume,Decision,Reason_for_decision,Job_Description
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...
...,...,...,...,...,...
10169,Product Manager,Here's a sample resume for Diana Miller:\n\n**...,reject,Unsatisfactory references or background check.,Here is a comprehensive job description for a ...
10170,UI Engineer,Here's a sample resume for Grace Taylor:\n\n**...,reject,Lack of relevant skills or experience.,Here is a sample job description for a UI Engi...
10171,UI Engineer,Here's a sample resume for Hank Brown:\n\n**Ha...,select,Growth mindset and adaptability.,Here is a job description for a UI Engineer ro...
10172,Data Engineer,Here's a sample resume for Diana Wilson:\n\n**...,reject,Lack of relevant skills or experience.,Here is a comprehensive job description for a ...


In [18]:
df.describe()

,Role,Resume,Decision,Reason_for_decision,Job_Description
count,10174,10174,10174,10174,10174
unique,45,10174,2,539,3446
top,Data Scientist,"Here's a sample resume for Charlie Miller, a P...",reject,Insufficient system design expertise for senio...,Join our team as a Product Manager and leverag...
freq,538,1,5114,730,103


In [19]:
import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s.,]', '', text)
    return text.lower().strip()

df['clean_resume'] = df['Resume'].apply(clean_text)
df['clean_jd'] = df['Job_Description'].apply(clean_text)

df[['clean_resume', 'clean_jd']].head(2)


,clean_resume,clean_jd
0,heres a professional resume for jason jones ja...,be part of a passionate team at the forefront ...
1,heres a professional resume for ann marshall a...,help us build the nextgeneration products as a...


In [20]:
label_map = {
    "select": 1,
    "reject": 0
}

df['label'] = df['Decision'].map(label_map)

df[['Decision', 'label']].head()

,Decision,label
0,reject,0
1,select,1
2,reject,0
3,select,1
4,reject,0


In [21]:
df['label'].value_counts()


,count
label,
0,5114
1,5060


In [22]:
df.head()

,Role,Resume,Decision,Reason_for_decision,Job_Description,clean_resume,clean_jd,label
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...,heres a professional resume for jason jones ja...,be part of a passionate team at the forefront ...,0
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...,heres a professional resume for ann marshall a...,help us build the nextgeneration products as a...,1
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...,heres a professional resume for patrick mcclai...,we need a human resources specialist to enhanc...,0
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...,heres a professional resume for patricia gray ...,be part of a passionate team at the forefront ...,1
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...,heres a professional resume for amanda gross a...,we are looking for an experienced ecommerce sp...,0


In [23]:
def extract_section(text, keywords, max_len=150):
    for key in keywords:
        if key in text:
            start = text.find(key)
            section = text[start:start + max_len]
            return section
    return ""

skills_keys =  [
    'skills',
    'technical skills',
    'core skills',
    'key skills',
    'technologies'
]
exp_keys =[
    'professional experience',
    'work experience',
    'experience',
    'employment history'
]
edu_keys =  [
    'education',
    'academic background',
    'qualifications',
    'degree'
]
cert_keys = ['certification', 'certifications']

df['skills_text'] = df['clean_resume'].apply(lambda x: extract_section(x, skills_keys))
df['experience_text'] = df['clean_resume'].apply(lambda x: extract_section(x, exp_keys))
df['education_text'] = df['clean_resume'].apply(lambda x: extract_section(x, edu_keys))
df['certificate_text']=df['clean_resume'].apply(lambda x:extract_section(x,cert_keys))
df.head()


,Role,Resume,Decision,Reason_for_decision,Job_Description,clean_resume,clean_jd,label,skills_text,experience_text,education_text,certificate_text
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...,heres a professional resume for jason jones ja...,be part of a passionate team at the forefront ...,0,skills inventory management seo for ecommerc...,"professional experience ecommerce specialist, ...",education bachelors degree in business admini...,certifications google analytics certification...
1,Game Developer,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...,heres a professional resume for ann marshall a...,help us build the nextgeneration products as a...,1,,experience in creating engaging game experienc...,,
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...,heres a professional resume for patrick mcclai...,we need a human resources specialist to enhanc...,0,"skills hris systems workday, bamboohr, etc. ...","work experience human resources generalist, xy...",education bachelors degree in human resources...,certifications shrmcp society for human resou...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...,heres a professional resume for patricia gray ...,be part of a passionate team at the forefront ...,1,skills product listing management seo for ec...,"work experience ecommerce specialist, abc onli...",education bachelors degree in business admini...,certification program certifications google a...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...,heres a professional resume for amanda gross a...,we are looking for an experienced ecommerce sp...,0,skills inventory management software tradege...,"work experience ecommerce specialist, abc onli...",education bachelors degree in business admini...,certifications google analytics certification...


In [24]:
print(df.iloc[1]["Resume"])


Here's a professional resume for Ann Marshall:

Ann Marshall
Contact Information:

* Email: [ann.marshall@email.com](mailto:ann.marshall@email.com)
* Phone: (123) 456-7890
* LinkedIn: linkedin.com/in/annmarshall
* GitHub: github.com/annmarshall

Summary:
Highly motivated and detail-oriented game developer with 5+ years of experience in creating engaging game experiences. Skilled in Unity and Unreal Engine, with expertise in C


In [25]:
def build_resume_text(row):
    sections = []

    if pd.notna(row['skills_text']) and row['skills_text'].strip():
        sections.append(f"[SKILLS] {row['skills_text']}")

    if pd.notna(row['experience_text']) and row['experience_text'].strip():
        sections.append(f"[EXPERIENCE] {row['experience_text']}")

    if pd.notna(row['education_text']) and row['education_text'].strip():
        sections.append(f"[EDUCATION] {row['education_text']}")

    if pd.notna(row['certificate_text']) and row['certificate_text'].strip():
        sections.append(f"[CERTIFICATIONS] {row['certificate_text']}")

    return " ".join(sections)


df['structured_resume'] = df.apply(build_resume_text, axis=1)

print(df['structured_resume'].iloc[0])

[SKILLS] skills  inventory management  seo for ecommerce  online advertising google ads, facebook ads  analytics google analytics, excel  data analysis  ecomme [EXPERIENCE] professional experience ecommerce specialist, xyz corporation 2018present  managed inventory levels across multiple channels, resulting in a 25 reduct [EDUCATION] education  bachelors degree in business administration, university name 2015 skills  inventory management  seo for ecommerce  online advertising googl [CERTIFICATIONS] certifications  google analytics certification  hubspot inbound marketing certification  shopify plus certification references available upon request.


In [26]:
train_df = pd.DataFrame({
    'text_a': df['structured_resume'],
    'text_b': df['clean_jd'],
    'label': df['label']
})

print(train_df.head())
print("Total training samples:", len(train_df))

                                              text_a  \
0  [SKILLS] skills  inventory management  seo for...   
1  [EXPERIENCE] experience in creating engaging g...   
2  [SKILLS] skills  hris systems workday, bambooh...   
3  [SKILLS] skills  product listing management  s...   
4  [SKILLS] skills  inventory management software...   

                                              text_b  label  
0  be part of a passionate team at the forefront ...      0  
1  help us build the nextgeneration products as a...      1  
2  we need a human resources specialist to enhanc...      0  
3  be part of a passionate team at the forefront ...      1  
4  we are looking for an experienced ecommerce sp...      0  
Total training samples: 10174


In [27]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    list(zip(train_df['text_a'], train_df['text_b'])),
    train_df['label'],
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)


In [28]:
lengths = []

for a, b in zip(train_df['text_a'], train_df['text_b']):
    tokens = tokenizer.encode(a, b)
    lengths.append(len(tokens))

print("Max:", max(lengths))
print("Average:", sum(lengths)/len(lengths))

Max: 1020
Average: 177.54826027127973


In [29]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_pairs(text_pairs):
    return tokenizer(
        [a for a, b in text_pairs],
        [b for a, b in text_pairs],
        padding=True,#pads sequences to the longest sequence in the current batch not necessarily to max-length
        truncation=True,
        max_length=512,
        return_tensors="pt"#without it the output will be python list
    )

train_encodings = tokenize_pairs(train_texts)
val_encodings = tokenize_pairs(val_texts)


In [30]:
hi

NameError: name 'hi' is not defined

In [31]:
train_encodings

{'input_ids': tensor([[ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 3325,  ...,    0,    0,    0],
        ...,
        [ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 3325,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

In [32]:
train_encodings.keys()

KeysView({'input_ids': tensor([[ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 3325,  ...,    0,    0,    0],
        ...,
        [ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 4813,  ...,    0,    0,    0],
        [ 101, 1031, 3325,  ...,    0,    0,    0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])})

In [37]:
import torch

class ResumeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx],dtype=torch.long)#Classification labels must be integers.of type torch.long
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ResumeDataset(train_encodings, train_labels)
val_dataset = ResumeDataset(val_encodings, val_labels)


In [38]:
sample = train_dataset[0]

for k, v in sample.items():
    print(k, v.shape if hasattr(v, "shape") else v)

input_ids torch.Size([512])
token_type_ids torch.Size([512])
attention_mask torch.Size([512])
labels torch.Size([])


In [39]:
train_dataset[0]

{'input_ids': tensor([  101,  1031,  4813,  1033,  4813,  1010,  2007,  1996,  3754,  2000,
          2147,  9174,  1998,  2004,  2112,  1997,  1037,  2136,  1012,  4180,
          3015,  3325,  4180,  3213,  1010,  5925,  5821,  4034,  2760, 28994,
          4765, 13675,  1031,  3325,  1033,  3325,  1999,  4526, 11973,  1998,
         12367,  8082,  4180,  2005,  2536,  6088,  1012, 10003,  2650,  2501,
          1997,  4852,  3784, 16476,  2083, 27457, 25697,  2378,  1031,  2495,
          1033,  2495,  5065,  2015,  3014,  1999,  2394,  1010,  1060,  2100,
          2480,  2118,  2325,  7378,  4180,  3006,  2121,  1010,  4180,  5821,
          2820, 10476, 10106,  3453,  1010,  5925,  5003,  1031, 10618,  2015,
          1033, 10618,  2015,  9594, 13102,  4140,  1999, 15494,  5821, 10618,
          2760,  8224, 25095, 10618, 10476,  7604,  2800,  2588,  5227,  1012,
          2023,  3252,  2019,   102,  2065,  2115,  2063, 13459,  2055,  4007,
          3330,  1010,  2057,  2342,  2

In [40]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2   # 0 = reject, 1 = select
)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [41]:
sample = train_dataset[0]

outputs = model(
    input_ids=sample['input_ids'].unsqueeze(0),
    attention_mask=sample['attention_mask'].unsqueeze(0),
    labels=sample['labels'].unsqueeze(0)
)

print(outputs)


SequenceClassifierOutput(loss=tensor(0.6833, grad_fn=<NllLossBackward0>), logits=tensor([[-0.3621, -0.3423]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


In [42]:
torch.argmax(outputs.logits, dim=1).item()


1

In [ ]:
!pip install -U transformers accelerate


In [ ]:
import transformers
print(transformers.__version__)


In [43]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",#Directory where checkpoints are stored. when erro occurs we can resume from a checkpoint
    eval_strategy="epoch",#evaluate after each epoch
    save_strategy="epoch",#save checkpoint after each epoch can recover best model using this
    per_device_train_batch_size=8,#8 Resumes=Update Weights 8=based on gpu
    per_device_eval_batch_size=8,#Batch size during validation.Usually same as train batch size.
    num_train_epochs=3,# to reduce overfitting
    learning_rate=2e-5,#contron weight updates
    weight_decay=0.01,#regularization to reduce overfitting
    logging_dir="./logs",#stores traing logs
    logging_steps=50,
    load_best_model_at_end=True
)


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [44]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='binary'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [45]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)


In [46]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.629041,0.627928,0.548894,0.524352,1.000000,0.687967
2,0.621748,0.623188,0.551843,0.938596,0.105731,0.190053
3,0.627360,0.622449,0.573464,0.588235,0.474308,0.525164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=3054, training_loss=0.6341567562542948, metrics={'train_runtime': 2543.857, 'train_samples_per_second': 9.598, 'train_steps_per_second': 1.201, 'total_flos': 6424382638725120.0, 'train_loss': 0.6341567562542948, 'epoch': 3.0})

trainer.evaluate() runs the model on the validation dataset in inference mode, computes the validation loss and any custom metrics defined in compute_metrics, and returns performance statistics

In [47]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.627360,0.622449,3,0.573464,0.588235,0.474308,0.525164


{'eval_loss': 0.6224489808082581,
 'eval_accuracy': 0.5734643734643735,
 'eval_precision': 0.5882352941176471,
 'eval_recall': 0.4743083003952569,
 'eval_f1': 0.5251641137855579}

In [48]:
device = model.device


In [49]:
sample = train_dataset[0]

inputs = {
    'input_ids': sample['input_ids'].unsqueeze(0).to(device),
    'attention_mask': sample['attention_mask'].unsqueeze(0).to(device),
    'labels': sample['labels'].unsqueeze(0).to(device)
}

outputs = model(**inputs)
print(outputs.logits)


tensor([[-0.0206, -0.0005]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [50]:
import torch

probs = torch.softmax(outputs.logits, dim=1)
print(probs)


tensor([[0.4950, 0.5050]], device='cuda:0', grad_fn=<SoftmaxBackward0>)


In [51]:
predicted_class = torch.argmax(probs, dim=1).item()

if predicted_class == 1:
    print("✅ Resume Selected")
else:
    print("❌ Resume Rejected")


✅ Resume Selected


In [52]:
predictions = trainer.predict(val_dataset)


In [53]:
import torch

logits = torch.tensor(predictions.predictions)
labels = torch.tensor(predictions.label_ids)


In [54]:
probs = torch.softmax(logits, dim=1)
predicted_classes = torch.argmax(probs, dim=1)
for i in range(10):
    print(
        f"Pred: {predicted_classes[i].item()}, "
        f"True: {labels[i].item()}, "
        f"Prob(Selected): {probs[i][1]:.3f}"
    )


Pred: 1, True: 1, Prob(Selected): 0.999
Pred: 0, True: 1, Prob(Selected): 0.494
Pred: 0, True: 1, Prob(Selected): 0.464
Pred: 1, True: 0, Prob(Selected): 0.503
Pred: 1, True: 1, Prob(Selected): 0.505
Pred: 0, True: 1, Prob(Selected): 0.485
Pred: 1, True: 0, Prob(Selected): 0.523
Pred: 1, True: 0, Prob(Selected): 0.527
Pred: 1, True: 1, Prob(Selected): 0.504
Pred: 0, True: 0, Prob(Selected): 0.499


In [55]:
def predict_match(resume_text, job_description):
    inputs = tokenizer(
        resume_text,
        job_description,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    return probs[0][1].item()  # probability of "Selected"


In [ ]:
def build_resume(skills_text="", experience_text="", education_text="", certificate_text=""):
    sections = []

    if skills_text and skills_text.strip():
        sections.append(f"Skills:\n{skills_text}")

    if experience_text and experience_text.strip():
        sections.append(f"Experience:\n{experience_text}")

    if education_text and education_text.strip():
        sections.append(f"Education:\n{education_text}")

    if certificate_text and certificate_text.strip():
        sections.append(f"Certifications:\n{certificate_text}")

    return "\n\n".join(sections)

In [ ]:
skills_text = """
Customer Service, Sales, Retail Operations
"""

experience_text = """
Worked in offline retail sales and handled customer interactions.
"""

education_text = """
Diploma in Retail Management.
"""
certificate_text = ""

jd_text = """
We are looking for an E-commerce Specialist with strong experience in
inventory management, SEO optimization, online advertising, and data analytics.
The candidate should have hands-on experience with Shopify or WooCommerce,
Google Analytics, and digital marketing tools.
"""
full_resume = build_resume(
    skills_text=skills_text,
    experience_text=experience_text,
    education_text=education_text,
    certificate_text=certificate_text
)

# Single prediction (same approach as training)
final_score = predict_match(full_resume, jd_text)

decision = "SELECTED" if final_score > 0.5 else "REJECTED"

print("Final Score:", final_score)
print("Decision:", decision)